# embed_only — BGE-768 embeddings

Standalone notebook. Reads raw `.json.gz` slices, embeds slice by slice,
and writes 4 compressed HDF5 files to `OUT_DIR`.

**Prerequisite**: run `preprocess_multi.ipynb` cells 1-5 first so the
`*_to_idx.json` index maps exist in `OUT_DIR`.

**Outputs**:
- `books_bge_base_768.h5`
- `reviews_bge_base_768.h5`
- `authors_bge_base_768.h5`
- `works_bge_base_768.h5`

## 1. Config

In [ ]:
from __future__ import annotations
import gc, gzip, json, os
from pathlib import Path
from typing import Iterator

import h5py
import numpy as np
import torch
from tqdm.auto import tqdm

RAW_DIR = Path("data_raw")
OUT_DIR = Path("data/processed_multi")   # must already contain *_to_idx.json

SLICES = ["poetry", "young_adult", "comics_graphic"]

def slice_books(s):   return RAW_DIR / f"goodreads_books_{s}.json.gz"
def slice_reviews(s): return RAW_DIR / f"goodreads_reviews_{s}.json.gz"

AUTHORS_GZ = RAW_DIR / "goodreads_book_authors.json.gz"
WORKS_GZ   = RAW_DIR / "goodreads_book_works.json.gz"

EMB_MODEL   = "BAAI/bge-base-en-v1.5"
EMB_DIM     = 768
EMB_BATCH   = 256    # GPU micro-batch
WRITE_BATCH = 2048   # rows per encode() call
MAX_CHARS   = 2000

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"OUT_DIR: {OUT_DIR.resolve()}")

for name in ["book_id_to_idx", "review_to_idx", "author_to_idx", "work_to_idx"]:
    p = OUT_DIR / f"{name}.json"
    assert p.exists(), f"Missing {p} -- run preprocess_multi.ipynb cells 1-5 first"
print("All idx maps found.")

def mem_gb() -> float:
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024**3
    except ImportError:
        return -1.0

def iter_jsonl_gz(path: Path) -> Iterator[dict]:
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for line in fh:
            yield json.loads(line)


## 2. Load idx maps

In [ ]:
book_to_idx   = json.loads((OUT_DIR / "book_id_to_idx.json").read_text(encoding="utf-8"))
review_to_idx = json.loads((OUT_DIR / "review_to_idx.json").read_text(encoding="utf-8"))
author_to_idx = json.loads((OUT_DIR / "author_to_idx.json").read_text(encoding="utf-8"))
work_to_idx   = json.loads((OUT_DIR / "work_to_idx.json").read_text(encoding="utf-8"))

book_ids_sorted   = sorted(book_to_idx,   key=book_to_idx.__getitem__)
review_ids_sorted = sorted(review_to_idx, key=review_to_idx.__getitem__)
author_ids_sorted = sorted(author_to_idx, key=author_to_idx.__getitem__)
work_ids_sorted   = sorted(work_to_idx,   key=work_to_idx.__getitem__)

print(f"books   : {len(book_to_idx):>10,}")
print(f"reviews : {len(review_to_idx):>10,}")
print(f"authors : {len(author_to_idx):>10,}")
print(f"works   : {len(work_to_idx):>10,}")


## 3. Model + helpers

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Loading {EMB_MODEL} on {device}...")
emb_model = SentenceTransformer(EMB_MODEL, device=device)
print("Model ready.")


def _make_memmap(abs_path: str, n: int) -> np.memmap:
    """Create a fresh writable memmap at an absolute path (avoids Windows Errno 22)."""
    return np.memmap(abs_path, dtype=np.float32, mode="w+", shape=(n, EMB_DIM))


def _embed_to_memmap(tmp: np.memmap, g_idx: list[int], texts: list[str], desc: str = "") -> None:
    """Encode texts in WRITE_BATCH chunks, write into memmap at positions g_idx."""
    for start in tqdm(range(0, len(texts), WRITE_BATCH), desc=desc, unit="batch"):
        end  = start + WRITE_BATCH
        embs = emb_model.encode(
            texts[start:end],
            batch_size=EMB_BATCH,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        ).astype(np.float32)
        tmp[g_idx[start:end]] = embs


def _save_h5(tmp: np.memmap, ids_sorted: list[str], id_key: str, h5_path: Path) -> None:
    """Write compressed HDF5 from memmap (one sequential pass), then delete tmp file."""
    tmp_file = tmp.filename
    print(f"  writing {h5_path.name}  shape={tmp.shape}  mem={mem_gb():.2f} GB")
    with h5py.File(h5_path, "w") as hf:
        hf.create_dataset(id_key, data=np.array(ids_sorted, dtype="S40"))
        hf.create_dataset("embedding", data=tmp[:],
                          compression="gzip", compression_opts=4)
    tmp.flush()
    del tmp; gc.collect()
    try:
        os.remove(tmp_file)
    except OSError:
        pass
    with h5py.File(h5_path, "r") as hf:
        norms = np.linalg.norm(hf["embedding"][:3], axis=1).round(4)
    print(f"  saved  {h5_path}  norm[0:3]={norms}")


## 4. Books — title + description (slice by slice)

Peak RAM = one slice of text strings + WRITE_BATCH embeddings.

In [ ]:
n_books  = len(book_ids_sorted)
abs_path = str((OUT_DIR / "_tmp_books.bin").resolve())
tmp      = _make_memmap(abs_path, n_books)

for s in SLICES:
    print(f"\n[{s}] streaming book texts  mem={mem_gb():.2f} GB")
    pairs: list[tuple[int, str]] = []
    for rec in tqdm(iter_jsonl_gz(slice_books(s)), desc=f"  {s}/stream"):
        bid = str(rec.get("book_id", ""))
        if bid in book_to_idx:
            t = f"{rec.get('title') or ''}. {rec.get('description') or ''}".strip()[:MAX_CHARS]
            pairs.append((book_to_idx[bid], t or "[no text]"))
    pairs.sort()
    g_idx = [p[0] for p in pairs]
    texts = [p[1] for p in pairs]
    del pairs; gc.collect()
    _embed_to_memmap(tmp, g_idx, texts, desc=f"  {s}/embed")
    del g_idx, texts; gc.collect()
    print(f"  {s} done  mem={mem_gb():.2f} GB")

_save_h5(tmp, book_ids_sorted, "book_id", OUT_DIR / "books_bge_base_768.h5")
print("\nBooks complete.")


## 5. Reviews — review_text (slice by slice)

Largest entity. Texts never all in RAM at once.

In [ ]:
n_reviews = len(review_ids_sorted)
abs_path  = str((OUT_DIR / "_tmp_reviews.bin").resolve())
tmp       = _make_memmap(abs_path, n_reviews)

for s in SLICES:
    print(f"\n[{s}] streaming review texts  mem={mem_gb():.2f} GB")
    pairs: list[tuple[int, str]] = []
    for rec in tqdm(iter_jsonl_gz(slice_reviews(s)), desc=f"  {s}/stream"):
        rid  = str(rec.get("review_id", ""))
        text = (rec.get("review_text") or "").strip()
        if rid in review_to_idx and text:
            pairs.append((review_to_idx[rid], text[:MAX_CHARS]))
    pairs.sort()
    g_idx = [p[0] for p in pairs]
    texts = [p[1] for p in pairs]
    del pairs; gc.collect()
    _embed_to_memmap(tmp, g_idx, texts, desc=f"  {s}/embed")
    del g_idx, texts; gc.collect()
    print(f"  {s} done  mem={mem_gb():.2f} GB")

_save_h5(tmp, review_ids_sorted, "review_id", OUT_DIR / "reviews_bge_base_768.h5")
print("\nReviews complete.")


## 6. Authors — name (global file, one pass)

In [ ]:
n_authors = len(author_ids_sorted)
abs_path  = str((OUT_DIR / "_tmp_authors.bin").resolve())
tmp       = _make_memmap(abs_path, n_authors)

pairs: list[tuple[int, str]] = []
for rec in tqdm(iter_jsonl_gz(AUTHORS_GZ), desc="authors/stream"):
    aid = str(rec.get("author_id", ""))
    if aid in author_to_idx:
        name = (rec.get("name") or "").strip()[:MAX_CHARS]
        pairs.append((author_to_idx[aid], name or "[unknown author]"))
pairs.sort()
g_idx = [p[0] for p in pairs]
texts = [p[1] for p in pairs]
del pairs; gc.collect()
print(f"  {len(texts):,} authors  mem={mem_gb():.2f} GB")
_embed_to_memmap(tmp, g_idx, texts, desc="authors/embed")
del g_idx, texts; gc.collect()

_save_h5(tmp, author_ids_sorted, "author_id", OUT_DIR / "authors_bge_base_768.h5")
print("\nAuthors complete.")


## 7. Works — original_title (global file, one pass)

In [ ]:
n_works  = len(work_ids_sorted)
abs_path = str((OUT_DIR / "_tmp_works.bin").resolve())
tmp      = _make_memmap(abs_path, n_works)

# Need book titles to fill in works with no original_title.
book_title_by_id: dict[str, str] = {}
for s in SLICES:
    for rec in iter_jsonl_gz(slice_books(s)):
        bid = str(rec.get("book_id", ""))
        if bid in book_to_idx:
            book_title_by_id[bid] = (rec.get("title") or "").strip()

pairs: list[tuple[int, str]] = []
for rec in tqdm(iter_jsonl_gz(WORKS_GZ), desc="works/stream"):
    wid = str(rec.get("work_id", ""))
    if wid in work_to_idx:
        t = (rec.get("original_title") or "").strip()
        if not t:
            best = str(rec.get("best_book_id") or "")
            t = book_title_by_id.get(best, "")
        pairs.append((work_to_idx[wid], (t or "[no title]")[:MAX_CHARS]))
del book_title_by_id
pairs.sort()
g_idx = [p[0] for p in pairs]
texts = [p[1] for p in pairs]
del pairs; gc.collect()
print(f"  {len(texts):,} works  mem={mem_gb():.2f} GB")
_embed_to_memmap(tmp, g_idx, texts, desc="works/embed")
del g_idx, texts; gc.collect()

_save_h5(tmp, work_ids_sorted, "work_id", OUT_DIR / "works_bge_base_768.h5")
print("\nWorks complete.")


## 8. Cleanup + verify

In [ ]:
del emb_model; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Verifying...")
checks = [
    ("books_bge_base_768.h5",   len(book_to_idx)),
    ("reviews_bge_base_768.h5", len(review_to_idx)),
    ("authors_bge_base_768.h5", len(author_to_idx)),
    ("works_bge_base_768.h5",   len(work_to_idx)),
]
for fname, n in checks:
    with h5py.File(OUT_DIR / fname, "r") as hf:
        emb = hf["embedding"]
        assert emb.shape == (n, EMB_DIM), f"{fname}: {emb.shape} != ({n},{EMB_DIM})"
        assert emb.dtype == np.float32
        norms = np.linalg.norm(emb[:5], axis=1)
        assert np.allclose(norms, 1.0, atol=1e-4), f"{fname}: not normalized"
    print(f"  {fname:<35} ({n:>8,}, {EMB_DIM})  OK")

print("\nAll done.")
